In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from pyspark.sql.functions import window, sum as spark_sum, count, col
import os, shutil, time
import pandas as pd
from datetime import datetime

spark = SparkSession.builder \
    .appName("Tugas10-KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

print("SparkSession siap. Versi Spark:", spark.version)

SparkSession siap. Versi Spark: 3.5.9


In [3]:
folder_stream = "tugas10_stream"
for f in [folder_stream, "tugas10_output", "tugas10_checkpoint"]:
    shutil.rmtree(f, ignore_errors=True)
os.makedirs(folder_stream, exist_ok=True)

skema_pesanan = StructType([
    StructField("pesanan_id", StringType(), True),
    StructField("nama_cabang", StringType(), True),
    StructField("nama_menu", StringType(), True),
    StructField("qty", IntegerType(), True),
    StructField("harga_satuan", IntegerType(), True),
    StructField("waktu_pesan", TimestampType(), True),
])

df_stream = spark.readStream \
    .format("csv") \
    .schema(skema_pesanan) \
    .option("header", True) \
    .load(folder_stream)

print("df_stream.isStreaming:", df_stream.isStreaming)

df_stream.isStreaming: True


In [4]:
df_stream = df_stream.withColumn("total_harga", col("qty") * col("harga_satuan"))

agregasi_per_cabang = df_stream.groupBy("nama_cabang").agg(
    spark_sum("total_harga").alias("total_penjualan"),
    count("pesanan_id").alias("jumlah_pesanan")
)

query_cabang = agregasi_per_cabang.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_cabang") \
    .start()

print("Query agregasi_cabang dimulai.")

Query agregasi_cabang dimulai.


In [5]:
df_stream_waktu = spark.readStream \
    .format("csv") \
    .schema(skema_pesanan) \
    .option("header", True) \
    .load(folder_stream) \
    .withColumn("total_harga", col("qty") * col("harga_satuan"))

agregasi_per_waktu = df_stream_waktu.groupBy(
    window(col("waktu_pesan"), "2 minutes")
).agg(
    spark_sum("total_harga").alias("total_penjualan")
)

query_waktu = agregasi_per_waktu.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_waktu") \
    .start()

print("Query agregasi_waktu dimulai.")

Query agregasi_waktu dimulai.


In [9]:
def kirim_batch(nama_file, baris):
    df = pd.DataFrame(baris, columns=["pesanan_id", "nama_cabang", "nama_menu", "qty", "harga_satuan"])
    df["waktu_pesan"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    df.to_csv(f"{folder_stream}/{nama_file}", index=False)

def tampilkan_cabang():
    spark.sql("SELECT * FROM agregasi_cabang ORDER BY nama_cabang").show()

# Batch 1
kirim_batch("batch_1.csv", [
    ["P001", "Magelang", "Kopi Susu", 2, 18000],
    ["P002", "Yogyakarta", "Americano", 1, 20000],
    ["P003", "Magelang", "Es Teh", 3, 8000],
])
print("Batch 1 dikirim")
tampilkan_cabang()

# Batch 2
kirim_batch("batch_2.csv", [
    ["P004", "Solo", "Latte", 2, 25000],
    ["P005", "Yogyakarta", "Kopi Susu", 4, 18000],
    ["P006", "Magelang", "Americano", 1, 20000],
])
print("Batch 2 dikirim")
tampilkan_cabang()

# Batch 3
kirim_batch("batch_3.csv", [
    ["P007", "Semarang", "Kopi Susu", 3, 18000],
    ["P008", "Solo", "Es Teh", 2, 8000],
    ["P009", "Yogyakarta", "Latte", 1, 25000],
])
print("Batch 3 dikirim")
tampilkan_cabang()

# Hasil window waktu (bonus untuk bagian C)
spark.sql("SELECT window.start, window.end, total_penjualan FROM agregasi_waktu ORDER BY window.start").show(truncate=False)

Batch 1 dikirim
+-----------+---------------+--------------+
|nama_cabang|total_penjualan|jumlah_pesanan|
+-----------+---------------+--------------+
|   Magelang|         130000|             4|
|   Semarang|          54000|             1|
|       Solo|          66000|             2|
| Yogyakarta|         117000|             3|
+-----------+---------------+--------------+

Batch 2 dikirim
+-----------+---------------+--------------+
|nama_cabang|total_penjualan|jumlah_pesanan|
+-----------+---------------+--------------+
|   Magelang|         130000|             4|
|   Semarang|          54000|             1|
|       Solo|          66000|             2|
| Yogyakarta|         117000|             3|
+-----------+---------------+--------------+

Batch 3 dikirim
+-----------+---------------+--------------+
|nama_cabang|total_penjualan|jumlah_pesanan|
+-----------+---------------+--------------+
|   Magelang|         130000|             4|
|   Semarang|          54000|             1|
|    

In [11]:
query3 = df_stream.writeStream \
    .format("parquet") \
    .option("path", "tugas10_output") \
    .option("checkpointLocation", "tugas10_checkpoint") \
    .outputMode("append") \
    .start()

kirim_batch("batch_4.csv", [
    ["P010", "Magelang", "Latte", 2, 25000],
])

In [12]:
!ls tugas10_output

spark.read.parquet("tugas10_output").show()

part-00000-f44f95c7-5c4a-4ed4-a4bf-7f28c62e1469-c000.snappy.parquet
part-00001-909ab5ea-f50b-454b-925a-7be2afc8e1bf-c000.snappy.parquet
part-00002-52babf3f-dc93-4d2f-b7c8-06c1d9c8e6ea-c000.snappy.parquet
part-00003-6d71a02b-9c3c-4c7f-acc7-0b0117d75bd0-c000.snappy.parquet
_spark_metadata
+----------+-----------+---------+---+------------+-------------------+-----------+
|pesanan_id|nama_cabang|nama_menu|qty|harga_satuan|        waktu_pesan|total_harga|
+----------+-----------+---------+---+------------+-------------------+-----------+
|      P001|   Magelang|Kopi Susu|  2|       18000|2026-11-05 09:07:30|      36000|
|      P002| Yogyakarta|Americano|  1|       20000|2026-11-05 09:07:30|      20000|
|      P003|   Magelang|   Es Teh|  3|        8000|2026-11-05 09:07:30|      24000|
|      P004|       Solo|    Latte|  2|       25000|2026-11-05 09:07:38|      50000|
|      P005| Yogyakarta|Kopi Susu|  4|       18000|2026-11-05 09:07:38|      72000|
|      P006|   Magelang|Americano|  1|  

In [13]:
for q in spark.streams.active:
    print("Menghentikan:", q.name)
    q.stop()
spark.stop()

Menghentikan: None
Menghentikan: agregasi_waktu
Menghentikan: agregasi_cabang
